# EDA Exploratorio - Sismos SSN-UNAM
Notebook complementario al pipeline. Carga la capa Silver y produce las gráficas exploratorias para el reporte.

In [ ]:
import os
import sys
from pathlib import Path


def detect_project_root():
    env_root = os.environ.get("SSN_PROJECT_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if candidate.exists():
            return candidate

    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "config" / "settings.py").exists():
            return candidate

    return current


ROOT = detect_project_root()
sys.path.insert(0, str(ROOT))
print(f"Proyecto detectado en: {ROOT}")

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    print("Estas en Colab. Si el repo vive en Google Drive, monta la unidad antes de seguir.")
    # drive.mount('/content/drive')

from config.settings import SILVER_PATH, GOLD_PATH
print(f"Silver: {SILVER_PATH}")
print(f"Gold: {GOLD_PATH}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# El setup de la celda anterior define ROOT, SILVER_PATH y GOLD_PATH.
df = pd.read_parquet(SILVER_PATH, engine='pyarrow')
print(f'Filas Silver: {len(df):,}')
df.head()

In [ ]:
# Distribucion de magnitudes
out_dir = ROOT / 'docs' / 'diagramas'
out_dir.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(10,5))
sns.histplot(df['magnitud'].dropna(), bins=50)
plt.title('Distribucion de magnitudes')
plt.xlabel('Magnitud')
plt.savefig(out_dir / 'hist_magnitudes.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Sismos por anio
out_dir = ROOT / 'docs' / 'diagramas'
out_dir.mkdir(parents=True, exist_ok=True)

anual = df.groupby('anio').size()
plt.figure(figsize=(12,5))
anual.plot()
plt.title('Sismos por anio 1900-2026')
plt.ylabel('Eventos')
plt.savefig(out_dir / 'sismos_por_anio.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Top 15 estados
out_dir = ROOT / 'docs' / 'diagramas'
out_dir.mkdir(parents=True, exist_ok=True)

top = df['estado'].value_counts().head(15)
plt.figure(figsize=(10,6))
top.plot(kind='barh')
plt.title('Top 15 estados con mas sismos')
plt.gca().invert_yaxis()
plt.savefig(out_dir / 'top_estados.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap hora-vs-mes
out_dir = ROOT / 'docs' / 'diagramas'
out_dir.mkdir(parents=True, exist_ok=True)

pivot = df.pivot_table(index='hora_del_dia', columns='mes', values='magnitud', aggfunc='count')
plt.figure(figsize=(12,6))
sns.heatmap(pivot, cmap='YlOrRd')
plt.title('Heatmap: sismos por hora del dia y mes')
plt.savefig(out_dir / 'heatmap_hora_mes.png', dpi=120, bbox_inches='tight')
plt.show()